# Pydantic格式的使用

## 1、基本使用

举例1：

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [2]:

from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名30岁的软件工程师")

print(result)
print(type(result))

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [3]:
print(f"姓名：{result.name}")
print(f"年龄：{result.age}")
print(f"职业：{result.occupation}")

姓名：张三
年龄：30
职业：软件工程师


举例2：

In [4]:

class MovieModel(BaseModel):
    """电影的详细信息"""
    title : str = Field(description="电影标题")
    year : int = Field(description="发行年份")
    director : str = Field(description="导演")
    rating : float = Field(description="电影评分，满分十分")


structured_model = model.with_structured_output(MovieModel)

result = structured_model.invoke("给出电影盗梦空间的信息")
print(result)

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=8.8


举例3：

In [5]:
from pydantic import BaseModel, Field
# 定义输出结构
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")


# ✅ v1.x：使用 with_structured_output
structured_model = model.with_structured_output(SentimentAnalysis)

# 调用
text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(
    f"分析以下文本的情感：\n{text}"
)

print(f"类型: {type(result)}")  # <class 'SentimentAnalysis'>
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

类型: <class '__main__.SentimentAnalysis'>
情感: positive
置信度: 0.99
关键词: ['实用', '学到了很多知识', '强烈推荐']


## 2、高级特性

### 2.1 情况1：可选字段

举例

In [19]:

from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

print(result)
print(type(result))

name='张三' age=0 occupation='软件工程师'
<class '__main__.Person'>


作为对比：

In [20]:

from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

print(result)
print(type(result))

name='张三' age=None occupation='软件工程师'
<class '__main__.Person'>


### 2.2 情况2：默认值

不同的模型供应商，对于此字段的支持是不同的。比如：closeai平台的gpt-5.4-mini模型就不支持此字段，而openrouter平台的gpt-5.4-mini模型就支持此字段。


closeai平台的模型：

In [21]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model_with_closeai = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

openrouter平台的模型：（1、需要充值  2、需要魔法）

In [22]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model_with_openrouter = ChatOpenRouter(
    model="openai/gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

举例1：使用closeai平台的模型

In [24]:

from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : int = Field(default=10,description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model_with_closeai.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

print(result)
print(type(result))

name='张三' age=20 occupation='软件工程师'
<class '__main__.Person'>


作为对比：使用openrouter平台的模型

In [25]:

from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : int = Field(default=10,description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model_with_openrouter.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

print(result)
print(type(result))

name='张三' age=10 occupation='软件工程师'
<class '__main__.Person'>


举例2：

In [26]:
class Config(BaseModel):
    timeout: Optional[int] = Field(30,description="超时时间(单位秒)")
    retry: bool = Field(False,description="是否支持重试")
    max_attempts: int = Field(6,description="最大重试次数")

# 测试
structured_llm = model_with_closeai.with_structured_output(Config)
structured_llm.invoke("配置要求：支持重试,最多重试5次")

Config(timeout=None, retry=True, max_attempts=5)

In [27]:
# 测试
structured_llm = model_with_openrouter.with_structured_output(Config)
structured_llm.invoke("配置要求：支持重试,最多重试5次")

Config(timeout=30, retry=True, max_attempts=5)

举例3：

In [28]:
from typing import Optional
from pydantic import BaseModel, Field

class Product(BaseModel):
    """产品信息"""
    name: str = Field(description="产品名称")
    price: float = Field(description="价格")
    description: Optional[str] = Field(description="产品描述")
    stock: int = Field(default=100, description="库存")


# 测试
structured_llm = model_with_openrouter.with_structured_output(Product)
print("\n场景1：完整信息")
result1 = structured_llm.invoke("iPhone 15 售价 5999 元，最新款智能手机，库存 50 台")
print(result1)

print("\n场景2：缺少描述和库存")
result2 = structured_llm.invoke("MacBook Pro 售价 12999 元")
print(result2)




场景1：完整信息
name='iPhone 15' price=5999.0 description='最新款智能手机' stock=50

场景2：缺少描述和库存
name='MacBook Pro' price=12999.0 description=None stock=100


### 2.3 情况3：枚举类型

举例：
方式1：

In [29]:
from enum import Enum


# 定义枚举类型
class Priority(str,Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"



class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

# print("\n提取结果：")
# print(f"  客户: {result.name}")
# print(f"  电话: {result.phone}")
# print(f"  邮箱: {result.email or '未提供'}")
# print(f"  问题: {result.issue}")
# print(f"  紧急程度: {result.urgency.value}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货，很着急' urgency=<Priority.HIGH: '高'>


方式2：

In [30]:

from typing import Literal


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低","中","高"] = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

# print("\n提取结果：")
# print(f"  客户: {result.name}")
# print(f"  电话: {result.phone}")
# print(f"  邮箱: {result.email or '未提供'}")
# print(f"  问题: {result.issue}")
# print(f"  紧急程度: {result.urgency.value}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency='高'


### 2.4 情况4：列表提取

举例1：

In [33]:

from typing import List


class Person(BaseModel):
    """人物信息"""
    name : str = Field(description="姓名")
    age :int = Field(description="年龄")

class PersonList(BaseModel):
    """人物列表"""
    people : List[Person]  # 多个Person的对象


structured_model = model.with_structured_output(PersonList)

result = structured_model.invoke("张三 30岁, 李四 40岁")
print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=40)]


举例2：

In [34]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = model.with_structured_output(Review)

review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")

print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


举例3：

In [35]:
class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")

# 测试
structured_llm = model.with_structured_output(Invoice)

invoice_text = """
发票号: INV-2024-001
日期: 2024-01-15
总金额: 1299.00
商品: MacBook Pro, AppleCare+
"""

invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

invoice_number='INV-2024-001' date='2024-01-15' total_amount=1299.0 items=['MacBook Pro', 'AppleCare+']


### 2.5 情况5：嵌套结构

举例1：

In [36]:

class Address(BaseModel):
    """地点描述"""
    city : str = Field(description="城市")
    district : str = Field(description="区域")


class Company(BaseModel):
    """公司信息"""
    name : str = Field(description="公司名称")
    address : Address = Field(description="公司所在地")

structured_model = model.with_structured_output(Company)

result = structured_model.invoke("阿里巴巴在杭州的滨江区")
print(result)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')


举例2：

In [37]:
from pydantic import BaseModel, Field
from typing import List

# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")  # 定义列表字段
    rating: float = Field(description="评分")

# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)

# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《盗梦空间》")

# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")

电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表: [Actor(name='莱昂纳多·迪卡普里奥', role='多姆·柯布'), Actor(name='约瑟夫·高登-莱维特', role='亚瑟'), Actor(name='艾伦·佩姬', role='阿里阿德涅'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='西里安·墨菲', role='罗伯特·费舍尔')]
评分: 8.8


举例3：

In [38]:
from pydantic import BaseModel
from typing import List

class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1-5")
    comment: str = Field(description="具体评价")

class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: str = Field(description="整体情感：positive/negative/neutral")
    overall_score: int = Field(description="综合评分，1-5")
    aspects: List[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")

# 创建结构化模型
structured_model = model.with_structured_output(ProductReview)
# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""

result = structured_model.invoke(
    f"分析以下产品评论：\n{review_text}"
)

print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")
for aspect in result.aspects:
    print(f"  - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\n总结: {result.summary}")

整体情感: positive
综合评分: 4/5

各维度评价:
  - 性能: 5/5 - 性能非常强大，运行大型软件毫无压力。
  - 屏幕: 5/5 - 屏幕色彩鲜艳，看视频很舒服。
  - 价格: 2/5 - 价格有点贵，性价比略受影响。
  - 噪音: 2/5 - 风扇噪音较大，使用时会有一定干扰。
  - 服务: 5/5 - 客服态度很好，物流也快，售后和配送体验不错。

总结: 整体评价偏正面，性能和屏幕表现突出，但价格偏高且风扇噪音较大。


### 2.6 情况6：限制条件

In [41]:
from pydantic import ValidationError

class User(BaseModel):
    name : str = Field(description="姓名",min_length=2,max_length=50)
    age : int = Field(description="年龄",le=150)
    email : str = Field(description="邮箱")


try:
    user1 = User(name="tom",age = 20,email="tom@126.com")
    print(f"[OK]{user1}")
except ValidationError as e:
    print(f"[FAIL]{e}")


[OK]name='tom' age=20 email='tom@126.com'


In [42]:
from pydantic import ValidationError

try:
    user2 = User(name="tom",age = 200,email="tom@126.com")
    print(f"[OK]{user2}")
except ValidationError as e:
    print(f"[FAIL]{e}")

[FAIL]1 validation error for User
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=200, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal


举例1：

使用closeai平台的模型

In [44]:
class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)


structured_model = model_with_closeai.with_structured_output(Product)
result = structured_model.invoke("华为mate 80 promax 价格是7999，当前库存100")
print(result)

name='华为mate 80 promax' price=7999.0 stock=100


In [45]:
result = structured_model.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
print(result)

name='华为mate 80 promax' price=7999.0 stock=100


使用openrouter平台的模型

In [46]:
class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)


structured_model = model_with_openrouter.with_structured_output(Product)
result = structured_model.invoke("华为mate 80 promax 价格是7999，当前库存100")
print(result)

name='华为mate 80 promax' price=7999.0 stock=100


In [47]:
result = structured_model.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
print(result)

name='华为mate 80 promax' price=1.0 stock=0
